# 79: Entries-Only with Danger Zone Exits

**Key Insight from Notebook 78:** "Never Exit" strategy beat everything else (844.9% vs 578.0% regime-adaptive).

## The Hypothesis:

James Check's **entry signals are excellent** but **exit signals fire too early**. Instead of trying to time every top, we should:

1. **Use Check's Buy The Dip entries** (4/5 conditions) - These work!
2. **Stay invested by default** - Don't exit on normal MVRV>2.0 signals
3. **Only exit in extreme danger zones** - True blow-off tops with -80% crash risk

## Danger Zone Detection (No Funding Data Needed!):

We'll use indicators available throughout Bitcoin's **full history (2009+)**:

| Indicator | Threshold | Why It Matters |
|-----------|-----------|----------------|
| Volatility | >100% annualized | Extreme chaos = unstable |
| 30-day ROC | >80% in 30 days | Parabolic unsustainable |
| Price vs MA200 | >3.0x | Extreme deviation from trend |
| MVRV | >4.0 | Historical euphoria levels |
| Funding Rate | >0.05% | *Optional* (only 2018+) |

**Danger Score:** Sum of above conditions. Exit if score ≥ 3/4 (or 4/5 if funding available).

## What We'll Test:

1. **Buy The Dip + Never Exit** - Baseline
2. **Buy The Dip + Danger Zone Exits** - New approach
3. **Buy The Dip + Original Exits (MVRV>2.0)** - Check framework
4. **Buy & Hold** - Benchmark

If "Danger Zone" beats "Never Exit", we've found the optimal strategy!

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    import vectorbt as vbt
    print("✓ VectorBT loaded")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "vectorbt"])
    import vectorbt as vbt

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Setup complete")

In [ ]:
# Configuration
PROJECT_ROOT = Path().resolve().parent
DATA_DIR = PROJECT_ROOT / "data" / "brk" / "daily"
GLASSNODE_DIR = PROJECT_ROOT / "data" / "glassnode" / "daily"

FEES = 0.001
SLIPPAGE = 0.001

## 1. Load Data (Full History 2009+)

In [ ]:
def load_metric(name: str, source: str = "brk") -> pd.Series:
    path = DATA_DIR / f"{name}.parquet" if source == "brk" else GLASSNODE_DIR / f"{name}.parquet"
    if not path.exists():
        return pd.Series(dtype=float)
    df = pd.read_parquet(path)
    if 'time' not in df.columns and isinstance(df.index, pd.DatetimeIndex):
        df = df.reset_index()
        if len(df.columns) == 2:
            df.columns = ['time', 'value']
    if 'value' not in df.columns:
        for col in df.columns:
            if col != 'time' and pd.api.types.is_numeric_dtype(df[col]):
                df['value'] = df[col]
                break
    if 'time' in df.columns and 'value' in df.columns:
        df['time'] = pd.to_datetime(df['time'])
        return df.set_index('time')['value'].sort_index()
    return pd.Series(dtype=float)

print("Loading data...")

metrics = {
    'price': ('price', 'brk'),
    'mvrv': ('mvrv', 'brk'),
    'mvrv_sth': ('mvrv_sth', 'brk'),
    'sopr_sth': ('sopr_sth', 'brk'),
    'sopr_lth': ('sopr_lth', 'brk'),
    'realized_profit': ('realized_profit', 'brk'),
    'realized_loss': ('realized_loss', 'brk'),
    'funding': ('funding_rate', 'glassnode'),  # Only 2018+
    'liq_long': ('liquidations_long', 'glassnode'),
    'liq_short': ('liquidations_short', 'glassnode'),
}

df_dict = {name: load_metric(metric, source) for name, (metric, source) in metrics.items()}
df = pd.DataFrame(df_dict).fillna(method='ffill')

print(f"✓ Data: {len(df)} days ({df.index[0].date()} to {df.index[-1].date()})")
print(f"  Price range: ${df['price'].min():.2f} to ${df['price'].max():.2f}")
print(f"  Funding data: {df['funding'].notna().sum()} days ({df['funding'].notna().sum() / len(df) * 100:.1f}%)")
df.head()

## 2. Calculate Danger Zone Indicators

These work on full history without funding data.

In [ ]:
print("Calculating danger zone indicators...\n")

# Price-based indicators (work on full history)
returns = df['price'].pct_change()
volatility = returns.rolling(30).std() * np.sqrt(365)
roc_30 = df['price'].pct_change(30)
ma_200 = df['price'].rolling(200).mean()
price_vs_ma = df['price'] / ma_200

# On-chain indicators (full history)
mvrv = df['mvrv']

# Derivatives (2018+ only)
funding_avg = df['funding'].rolling(30).mean()

# Summary
print("Indicator Summary (Full History):")
print("="*70)
print(f"Volatility:       {volatility.mean():.1%} avg, {volatility.max():.1%} max")
print(f"30-day ROC:       {roc_30.mean():.1%} avg, {roc_30.max():.1%} max")
print(f"Price vs MA200:   {price_vs_ma.mean():.2f}x avg, {price_vs_ma.max():.2f}x max")
print(f"MVRV:             {mvrv.mean():.2f} avg, {mvrv.max():.2f} max")
print(f"\nFunding Rate (2018+):")
print(f"  Available:      {funding_avg.notna().sum()} days")
print(f"  Average:        {funding_avg.mean():.4f}")
print(f"  Max:            {funding_avg.max():.4f}")

## 3. Define Danger Zone Scoring System

In [ ]:
def calculate_danger_score(df: pd.DataFrame, funding_available: pd.Series) -> pd.Series:
    """
    Calculate danger zone score (0-1 scale).
    
    Criteria (all periods):
    1. Volatility > 100% annualized (0.25 points)
    2. 30-day ROC > 80% (0.25 points)
    3. Price > 3.0x MA200 (0.25 points)
    4. MVRV > 4.0 (0.25 points)
    5. Funding > 0.05% (0.25 points, only when available)
    
    Exit when score >= 0.60 (3/5 conditions met)
    """
    # Price-based (always available)
    vol_danger = (volatility > 1.00).astype(float) * 0.25
    roc_danger = (roc_30 > 0.80).astype(float) * 0.25
    ma_danger = (price_vs_ma > 3.0).astype(float) * 0.25
    mvrv_danger = (df['mvrv'] > 4.0).astype(float) * 0.25
    
    # Funding-based (conditional)
    funding_danger = pd.Series(0.0, index=df.index)
    funding_danger[funding_available] = (funding_avg[funding_available] > 0.05).astype(float) * 0.25
    
    # Total danger score
    danger_score = vol_danger + roc_danger + ma_danger + mvrv_danger + funding_danger
    
    return danger_score


# Calculate danger scores
funding_available = df['funding'].notna()
danger_score = calculate_danger_score(df, funding_available)

print("Danger Zone Statistics:")
print("="*70)
print(f"Average danger score: {danger_score.mean():.2f}")
print(f"Max danger score:     {danger_score.max():.2f}")
print(f"Days in danger zone (score >= 0.60): {(danger_score >= 0.60).sum()} ({(danger_score >= 0.60).sum() / len(df) * 100:.1f}%)")
print(f"Days in extreme danger (score >= 0.75): {(danger_score >= 0.75).sum()} ({(danger_score >= 0.75).sum() / len(df) * 100:.1f}%)")

# Show top danger periods
print("\nTop 10 Most Dangerous Days:")
print("="*70)
top_danger = danger_score.nlargest(10)
for date, score in top_danger.items():
    price = df.loc[date, 'price']
    mvrv_val = df.loc[date, 'mvrv']
    print(f"{date.date()}: Score {score:.2f} | Price ${price:,.0f} | MVRV {mvrv_val:.2f}")

## 4. Generate Entry Signals (Same as Before)

In [ ]:
# Entry: Buy The Dip (4/5 conditions)
print("Generating entry signals...\n")

c1 = df['mvrv_sth'] < 1.0
c2 = df['sopr_sth'] < 1.0
c3 = (df['realized_profit'] / df['realized_loss']) < 1.0
c4 = df['funding'] <= 0.0  # Use when available, otherwise False
c5 = (df['liq_long'] / df['liq_short']) > 1.0  # Use when available

# Handle NaN values (early periods without funding/liq data)
c4 = c4.fillna(False)
c5 = c5.fillna(False)

entry_count = c1.astype(int) + c2.astype(int) + c3.astype(int) + c4.astype(int) + c5.astype(int)
entries = (entry_count >= 4).fillna(False).astype(bool)

print(f"✓ Entry signals: {entries.sum()}")
print(f"  First entry: {entries[entries].index[0].date()}")
print(f"  Last entry:  {entries[entries].index[-1].date()}")

# Entry frequency by year
print("\nEntry signals by year:")
for year in range(df.index[0].year, df.index[-1].year + 1):
    year_entries = entries[entries.index.year == year].sum()
    if year_entries > 0:
        print(f"  {year}: {year_entries}")

## 5. Generate Exit Signals (Multiple Strategies)

In [ ]:
print("Generating exit signals...\n")

# 1. Never exit
exits_never = pd.Series(False, index=df.index, dtype=bool)

# 2. Danger zone exits (score >= 0.60)
exits_danger_60 = (danger_score >= 0.60).fillna(False).astype(bool)

# 3. Extreme danger exits (score >= 0.75)
exits_danger_75 = (danger_score >= 0.75).fillna(False).astype(bool)

# 4. Original Check framework (MVRV>2.0 AND LTH-SOPR>1.5)
exits_original = ((df['mvrv'] > 2.0) & (df['sopr_lth'] > 1.5)).fillna(False).astype(bool)

# 5. Conservative danger (score >= 0.50)
exits_danger_50 = (danger_score >= 0.50).fillna(False).astype(bool)

print("Exit signal counts:")
print("="*70)
print(f"Never Exit:                 {exits_never.sum()}")
print(f"Danger Zone (≥0.50):        {exits_danger_50.sum()}")
print(f"Danger Zone (≥0.60):        {exits_danger_60.sum()}")
print(f"Extreme Danger (≥0.75):     {exits_danger_75.sum()}")
print(f"Original (MVRV>2.0):        {exits_original.sum()}")

# Verify dtypes
print("\nDtype verification:")
print(f"  entries: {entries.dtype}")
print(f"  exits_never: {exits_never.dtype}")
print(f"  exits_danger_60: {exits_danger_60.dtype}")
print(f"  exits_original: {exits_original.dtype}")

## 6. Visualize Danger Zones Over Time

In [ ]:
# Plot danger score over time
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Price with danger zones
ax1 = axes[0]
ax1.plot(df.index, df['price'], color='black', linewidth=2, label='BTC Price')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(danger_score >= 0.75), alpha=0.3, color='red', label='Extreme Danger (≥0.75)')
ax1.fill_between(df.index, 0, df['price'].max() * 1.2,
                  where=(danger_score >= 0.60) & (danger_score < 0.75), alpha=0.2, color='orange', label='Danger Zone (≥0.60)')
ax1.set_ylabel('BTC Price ($)', fontsize=12)
ax1.set_title('Danger Zone Detection Over Time', fontsize=14, fontweight='bold')
ax1.set_yscale('log')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Danger score
ax2 = axes[1]
ax2.plot(df.index, danger_score, color='red', linewidth=1.5, label='Danger Score')
ax2.axhline(0.60, color='orange', linestyle='--', alpha=0.7, label='Exit Threshold (0.60)')
ax2.axhline(0.75, color='red', linestyle='--', alpha=0.7, label='Extreme Threshold (0.75)')
ax2.fill_between(df.index, 0, 1, where=(danger_score >= 0.60), alpha=0.2, color='red')
ax2.set_ylabel('Danger Score', fontsize=12)
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylim(0, 1)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Red shading = Extreme danger (exit immediately)")
print("📊 Orange shading = Danger zone (consider exiting)")

## 7. Backtest All Strategies

In [ ]:
def backtest_strategy(df: pd.DataFrame, entries: pd.Series, exits: pd.Series, name: str) -> dict:
    """Run backtest using VectorBT."""
    pf = vbt.Portfolio.from_signals(
        close=df['price'],
        entries=entries,
        exits=exits,
        fees=FEES,
        slippage=SLIPPAGE,
        init_cash=10000,
        freq='1D'
    )
    return {
        'name': name,
        'portfolio': pf,
        'total_return': pf.total_return() * 100,
        'sharpe': pf.sharpe_ratio(),
        'max_dd': pf.max_drawdown() * 100,
        'num_trades': pf.trades.count(),
        'win_rate': pf.trades.win_rate() * 100 if pf.trades.count() > 0 else 0,
    }

print("\n" + "="*90)
print("BACKTESTING: ENTRIES-ONLY WITH DANGER ZONE EXITS")
print("="*90)

# Backtest all strategies
results = {}
results['never'] = backtest_strategy(df, entries, exits_never, "Never Exit")
results['danger_50'] = backtest_strategy(df, entries, exits_danger_50, "Danger Zone (≥0.50)")
results['danger_60'] = backtest_strategy(df, entries, exits_danger_60, "Danger Zone (≥0.60)")
results['danger_75'] = backtest_strategy(df, entries, exits_danger_75, "Extreme Danger (≥0.75)")
results['original'] = backtest_strategy(df, entries, exits_original, "Original (MVRV>2.0)")

# Buy and hold
bh_return = (df['price'].iloc[-1] / df['price'].iloc[0] - 1) * 100

# Results table
print(f"\n{'Strategy':<30} {'Return':>12} {'Sharpe':>8} {'Max DD':>10} {'Trades':>8} {'Win Rate':>10}")
print("-"*90)

for key, res in results.items():
    print(f"{res['name']:<30} {res['total_return']:>11.1f}% {res['sharpe']:>8.2f} {res['max_dd']:>9.1f}% {int(res['num_trades']):>8} {res['win_rate']:>9.1f}%")

print(f"{'Buy & Hold':<30} {bh_return:>11.1f}% {'~1.0':>8} {'?':>10} {'-':>8} {'-':>10}")
print("="*90)

# Comparison
print("\nVS BUY & HOLD:")
for key, res in results.items():
    diff = res['total_return'] - bh_return
    status = "✅ BEAT" if diff > 0 else "❌ LOST"
    print(f"  {res['name']:<30} {status} by {abs(diff):.1f}%")

# Best strategy
best = max(results.values(), key=lambda x: x['total_return'])
print(f"\n🏆 BEST STRATEGY: {best['name']} at {best['total_return']:.1f}%")

## 8. Equity Curves Comparison

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(16, 8))

# Buy & Hold
bh_equity = (df['price'] / df['price'].iloc[0]) * 10000
ax.plot(bh_equity.index, bh_equity.values, label='Buy & Hold', linewidth=3, color='black', linestyle='--', alpha=0.7)

# Strategies
colors = {'never': 'green', 'danger_50': 'blue', 'danger_60': 'orange', 'danger_75': 'red', 'original': 'gray'}
for key, res in results.items():
    equity = res['portfolio'].value()
    ax.plot(equity.index, equity.values, label=res['name'], linewidth=2, alpha=0.8, color=colors[key])

ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Portfolio Value ($)', fontsize=12)
ax.set_title('Entries-Only with Danger Zone Exits: Equity Curves', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal Portfolio Values:")
print(f"  Buy & Hold: ${bh_equity.iloc[-1]:,.0f}")
for key, res in results.items():
    final = res['portfolio'].value().iloc[-1]
    print(f"  {res['name']}: ${final:,.0f}")

## 9. Performance by Time Period

In [ ]:
# Split by major periods
periods = [
    ('2013-01-01', '2015-12-31', '2013-2015 (Early)'),
    ('2017-01-01', '2018-12-31', '2017-2018 Cycle'),
    ('2020-01-01', '2022-12-31', '2020-2022 Cycle'),
    ('2023-01-01', '2026-01-22', '2023-2026 Bull'),
]

print("\n" + "="*100)
print("PERFORMANCE BY TIME PERIOD")
print("="*100)

for start, end, label in periods:
    period_df = df[(df.index >= start) & (df.index <= end)].copy()
    if len(period_df) < 30:
        continue
    
    period_entries = entries[(entries.index >= start) & (entries.index <= end)]
    
    # Buy and hold
    bh = (period_df['price'].iloc[-1] / period_df['price'].iloc[0] - 1) * 100
    
    # Test danger zone strategies
    period_results = {}
    for key, name in [('never', 'Never Exit'), ('danger_60', 'Danger ≥0.60'), ('original', 'Original')]:
        if key == 'never':
            period_exits = exits_never[(exits_never.index >= start) & (exits_never.index <= end)]
        elif key == 'danger_60':
            period_exits = exits_danger_60[(exits_danger_60.index >= start) & (exits_danger_60.index <= end)]
        else:
            period_exits = exits_original[(exits_original.index >= start) & (exits_original.index <= end)]
        
        try:
            pf = vbt.Portfolio.from_signals(
                close=period_df['price'], entries=period_entries, exits=period_exits,
                fees=FEES, slippage=SLIPPAGE, init_cash=10000, freq='1D'
            )
            period_results[key] = pf.total_return() * 100
        except:
            period_results[key] = 0
    
    # Danger zone activity
    period_danger = danger_score[(danger_score.index >= start) & (danger_score.index <= end)]
    danger_days = (period_danger >= 0.60).sum()
    
    print(f"\n{label}:")
    print(f"  Danger zone days: {danger_days} ({danger_days / len(period_df) * 100:.1f}%)")
    print(f"  Buy & Hold: {bh:+.1f}%")
    for key, ret in period_results.items():
        name = 'Never Exit' if key == 'never' else ('Danger ≥0.60' if key == 'danger_60' else 'Original')
        print(f"  {name}: {ret:+.1f}% ({ret - bh:+.1f}% vs B&H)")
    
    # Winner
    best_key = max(period_results, key=period_results.get)
    best_name = 'Never Exit' if best_key == 'never' else ('Danger ≥0.60' if best_key == 'danger_60' else 'Original')
    print(f"  🏆 Winner: {best_name}")

print("\n" + "="*100)

## 10. Final Verdict

In [ ]:
print("\n" + "="*90)
print("FINAL VERDICT: ENTRIES-ONLY WITH DANGER ZONE EXITS")
print("="*90)

never = results['never']
danger_60 = results['danger_60']
original = results['original']
best = max(results.values(), key=lambda x: x['total_return'])

print(f"\n1. PERFORMANCE COMPARISON:")
print(f"   Buy & Hold:          {bh_return:.1f}%")
print(f"   Never Exit:          {never['total_return']:.1f}% ({never['total_return'] - bh_return:+.1f}%)")
print(f"   Danger Zone (≥0.60): {danger_60['total_return']:.1f}% ({danger_60['total_return'] - bh_return:+.1f}%)")
print(f"   Original (MVRV>2.0): {original['total_return']:.1f}% ({original['total_return'] - bh_return:+.1f}%)")
print(f"   Best: {best['name']} at {best['total_return']:.1f}%")

improvement = danger_60['total_return'] - never['total_return']
print(f"\n2. DANGER ZONE BENEFIT:")
print(f"   Danger Zone vs Never Exit: {improvement:+.1f}%")

if improvement > 10:
    print(f"   ✅ Danger zone exits add significant value!")
elif improvement > 0:
    print(f"   ✓ Modest benefit from danger zone exits")
else:
    print(f"   ❌ Danger zone exits hurt performance")

print(f"\n3. RISK-ADJUSTED PERFORMANCE:")
print(f"   Never Exit:     Sharpe {never['sharpe']:.2f}, DD {never['max_dd']:.1f}%")
print(f"   Danger ≥0.60:   Sharpe {danger_60['sharpe']:.2f}, DD {danger_60['max_dd']:.1f}%")
print(f"   Original:       Sharpe {original['sharpe']:.2f}, DD {original['max_dd']:.1f}%")

if danger_60['sharpe'] > never['sharpe']:
    print(f"   ✅ Danger zone has better risk-adjusted returns!")
else:
    print(f"   Never Exit has better Sharpe ratio")

print(f"\n4. TRADE STATISTICS:")
print(f"   Never Exit:   {int(never['num_trades'])} trades, {never['win_rate']:.1f}% win rate")
print(f"   Danger ≥0.60: {int(danger_60['num_trades'])} trades, {danger_60['win_rate']:.1f}% win rate")
print(f"   Original:     {int(original['num_trades'])} trades, {original['win_rate']:.1f}% win rate")

print(f"\n5. RECOMMENDATION:")

if best['name'] == 'Never Exit':
    print(f"   🎯 Optimal Strategy: Use Check's entries, NEVER EXIT")
    print(f"   📝 Implication: Exit signals don't add value in any regime")
    print(f"   💡 Why: Bitcoin's long-term trend dominates all timing attempts")
elif best['name'].startswith('Danger Zone'):
    threshold = best['name'].split('≥')[1].rstrip(')')
    print(f"   🎯 Optimal Strategy: Use Check's entries + Danger Zone exits (≥{threshold})")
    print(f"   📝 Exit only in extreme conditions (multiple warnings firing)")
    print(f"   ✅ This protects against -80% crashes while staying invested")
else:
    print(f"   🎯 Optimal Strategy: {best['name']}")
    print(f"   📝 Original Check framework works best")

# Final insight
if danger_60['total_return'] > bh_return:
    print(f"\n   🏆 SUCCESS: This strategy BEATS buy-and-hold!")
else:
    gap = bh_return - danger_60['total_return']
    print(f"\n   ⚠️  Still trails buy-and-hold by {gap:.1f}%")
    if danger_60['sharpe'] > 1.0:
        print(f"   But has superior risk-adjusted returns (Sharpe {danger_60['sharpe']:.2f})")

print("\n" + "="*90)

## Conclusion

This notebook tests whether "danger zone exits" (only exiting in extreme conditions) can improve upon "never exit" strategy.

**Key Insights:**
1. Danger zone detection works on full Bitcoin history (no funding data needed)
2. Uses price-based indicators available since 2009
3. Tests if protecting against -80% crashes adds value vs staying fully invested

**Next Steps:**
- If "Never Exit" wins → Stop using exit signals entirely
- If "Danger Zone" wins → Implement for live trading with real-time scoring
- Consider hybrid approach: 50% never exit, 50% danger zone exits